Copied this out of an AI generated template from Anaconda, a sort of pre-made environment thing that I thought might help. It seems promising enough, but I'd say we just use it as an example at best.

In [1]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
import matplotlib.pyplot as plt

class BB84Protocol:
    """
    BB84 Quantum Key Distribution Protocol Implementation
    
    Steps:
    1. Alice generates random bits and bases
    2. Alice encodes qubits and sends to Bob
    3. Bob measures with random bases
    4. Alice and Bob compare bases (classical channel)
    5. They keep bits where bases matched
    6. Error checking and privacy amplification
    """
    
    def __init__(self, num_bits=100):
        self.num_bits = num_bits
        self.alice_bits = []
        self.alice_bases = []
        self.bob_bases = []
        self.bob_results = []
        self.shared_key = []
        
    def alice_prepare_qubits(self):
        """Alice generates random bits and bases, then prepares qubits"""
        self.alice_bits = np.random.randint(0, 2, self.num_bits)
        self.alice_bases = np.random.randint(0, 2, self.num_bits)  # 0: Z-basis, 1: X-basis
        print(f"Alice's bits: {self.alice_bits[:20]}...")
        print(f"Alice's bases: {self.alice_bases[:20]}...")
        
    def bob_choose_bases(self):
        """Bob randomly chooses measurement bases"""
        self.bob_bases = np.random.randint(0, 2, self.num_bits)
        print(f"Bob's bases: {self.bob_bases[:20]}...")
        
    def simulate_quantum_channel(self, error_rate=0.0):
        """Simulate sending qubits from Alice to Bob"""
        simulator = AerSimulator()
        
        for i in range(self.num_bits):
            # Create quantum circuit
            qc = QuantumCircuit(1, 1)
            
            # Alice encodes her bit
            if self.alice_bits[i] == 1:
                qc.x(0)  # Flip to |1⟩
                
            # Alice applies basis rotation
            if self.alice_bases[i] == 1:  # X-basis
                qc.h(0)
                
            # Simulate channel noise (optional)
            if np.random.random() < error_rate:
                qc.x(0)  # Bit flip error
                
            # Bob measures in his chosen basis
            if self.bob_bases[i] == 1:  # X-basis
                qc.h(0)
                
            qc.measure(0, 0)
            
            # Execute circuit
            compiled_circuit = transpile(qc, simulator)
            result = simulator.run(compiled_circuit, shots=1).result()
            counts = result.get_counts()
            measured_bit = int(list(counts.keys())[0])
            self.bob_results.append(measured_bit)
            
        print(f"Bob's measurements: {self.bob_results[:20]}...")
        
    def sift_key(self):
        """Alice and Bob compare bases and keep matching ones"""
        for i in range(self.num_bits):
            if self.alice_bases[i] == self.bob_bases[i]:
                self.shared_key.append(self.bob_results[i])
                
        print(f"\nBases matched: {len(self.shared_key)}/{self.num_bits}")
        print(f"Shared key (first 20): {self.shared_key[:20]}...")
        
    def estimate_error_rate(self, sample_size=None):
        """Estimate quantum bit error rate (QBER)"""
        if sample_size is None:
            sample_size = min(len(self.shared_key) // 2, 50)
            
        errors = 0
        alice_sample = []
        
        # Get Alice's bits that correspond to shared key
        key_index = 0
        for i in range(self.num_bits):
            if self.alice_bases[i] == self.bob_bases[i]:
                if key_index < sample_size:
                    alice_sample.append(self.alice_bits[i])
                    if self.alice_bits[i] != self.shared_key[key_index]:
                        errors += 1
                key_index += 1
                
        qber = errors / sample_size if sample_size > 0 else 0
        print(f"\nQBER: {qber:.2%} ({errors}/{sample_size} errors)")
        
        # Remove sampled bits from key
        self.shared_key = self.shared_key[sample_size:]
        return qber
        
    def run_protocol(self, error_rate=0.0):
        """Execute the complete BB84 protocol"""
        print("=" * 50)
        print("BB84 Quantum Key Distribution Protocol")
        print("=" * 50)
        
        self.alice_prepare_qubits()
        self.bob_choose_bases()
        self.simulate_quantum_channel(error_rate)
        self.sift_key()
        qber = self.estimate_error_rate()
        
        print(f"\nFinal shared key length: {len(self.shared_key)} bits")
        return self.shared_key, qber


# Example usage
if __name__ == "__main__":
    # Run BB84 protocol
    bb84 = BB84Protocol(num_bits=200)
    shared_key, qber = bb84.run_protocol(error_rate=0.05)
    
    print(f"\n✓ Protocol complete!")
    print(f"✓ Secure key established: {len(shared_key)} bits")

BB84 Quantum Key Distribution Protocol
Alice's bits: [1 1 0 1 1 1 0 1 1 1 1 1 0 0 0 0 1 1 0 1]...
Alice's bases: [1 0 1 0 1 1 1 0 0 1 0 1 0 1 0 0 0 0 0 1]...
Bob's bases: [1 1 0 0 0 1 0 0 0 1 1 1 0 0 1 0 1 1 0 1]...
Bob's measurements: [1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1]...

Bases matched: 96/200
Shared key (first 20): [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1]...

QBER: 2.08% (1/48 errors)

Final shared key length: 48 bits

✓ Protocol complete!
✓ Secure key established: 48 bits
